# Assembling the master modeling dataset 

To-do: 
- remove hardcoded API key 

In [3]:
import pandas as pd
import requests
import geopandas as gpd
from shapely.geometry import Polygon, Point
import numpy as np
from functools import reduce

/hpc/m3/python/3.11.11/data_science-2025.08.21/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [10]:
# setting wd 
import os
os.chdir('/users/bkung/fooddesertproject')

In [41]:
# reading in EJI data file from CSV 
EJI_data = pd.read_csv('modeling_data/EJI_2024_United_States.csv')

In [42]:
# filtering data to relevant counties: Bexar, Dallas, Tarrant, Travis, Harris
EJI_texas = EJI_data[EJI_data['STATEFP'] == 48]
EJI_relevant_counties = EJI_texas[EJI_texas['COUNTYFP'].isin([29, 453, 201, 113, 439])] # Bexar, Travis, Harris, Dallas, Tarrant

In [43]:
# filtering to relevant columns 
modeling_EJI_data = EJI_relevant_counties[
    ['COUNTY', 'GEOID', 'TRACTCE', 'E_WLKIND', 'EPL_WLKIND', 'E_TOTPOP', 
     'M_TOTPOP', 'E_POV200', 'E_NOHSDP', 'E_UNINSUR', 'E_CHD', 
     'E_DIABETES', 'E_AFAM','E_ASIAN', 'E_HISP']
    ]

In [44]:
# renaming columns 
modeling_EJI_data = modeling_EJI_data.rename(
    columns={'TRACTCE': 'tract', 'E_WLKIND': 'walking_ind', 'EPL_WLKIND':'walkind_inv_perc', 'E_TOTPOP': 'total_pop', 'M_TOTPOP': 'total_pop_moe', 
             'E_POV200': 'below_200_fed_poverty_percentage', 'E_NOHSDP':'no_hs_diploma', 'E_UNINSUR': 'uninsured'})


In [45]:
# splitting up by county to facilitate final merges 
def separate_df(df, column, target_value, exclude=False):
    if exclude:
        df_separated = df[df[column] != target_value].reset_index(drop=True).copy()
    else:
        df_separated = df[df[column] == target_value].reset_index(drop=True).copy()
    return df_separated

bexar_EJI = separate_df(modeling_EJI_data, 'COUNTY', 'Bexar County')
travis_EJI = separate_df(modeling_EJI_data, 'COUNTY', 'Travis County')
harris_EJI = separate_df(modeling_EJI_data, 'COUNTY', 'Harris County')
dallas_EJI = separate_df(modeling_EJI_data, 'COUNTY', 'Dallas County')
tarrant_EJI = separate_df(modeling_EJI_data, 'COUNTY', 'Tarrant County')

In [47]:
# RETRIEVING ACS MEDIAN AGE DATA FROM API
# hard coding parameters (to change later)
API_KEY = "3260419d0dd1c45a470edaf688621f89b71fa441"
STATE_FIPS = "48"    # Texas
COUNTY_LIST = ["029", "453", "201", "113", "439"] 

# median age (total population) endpoint 
URL_BASE = "https://api.census.gov/data/2024/acs/acs5/profile"
VARIABLES = "NAME,DP05_0018E"

all_tracts_data = []

for county_fips in COUNTY_LIST:
    # building url
    url = f"{URL_BASE}?get={VARIABLES}&for=tract:*&in=state:{STATE_FIPS}+county:{county_fips}&key={API_KEY}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        # verifying API returned JSON first 
        content_type = response.headers.get('Content-Type', '')
        if 'application/json' in content_type:
            json_data = response.json()
            
            headers = json_data[0]
            rows = json_data[1:]
            county_df = pd.DataFrame(rows, columns=headers)
            all_tracts_data.append(county_df)
        else:
            # get text error if returned
            print(f"\n[Census API Error] Server returned text instead of data for county {county_fips}:")
            print(response.text.strip())

    except requests.exceptions.RequestException as e:
        print(f"Network or HTTP error for county {county_fips}: {e}")

# combining results 
if all_tracts_data:
    final_df = pd.concat(all_tracts_data, ignore_index=True)
    
    # converting and cleaning results
    final_df["DP05_0018E"] = pd.to_numeric(final_df["DP05_0018E"], errors='coerce')
    final_df = final_df.rename(columns={"DP05_0018E": "median_age"})
    
    print("\n--- SUCCESS! First 5 rows: ---")
    print(final_df[["NAME", "tract", "median_age"]].head())
else:
    print("\nNo data was collected. Read the API text errors above to see why.")



--- SUCCESS! First 5 rows: ---
                                     NAME   tract  median_age
0  Census Tract 1101; Bexar County; Texas  110100        34.6
1  Census Tract 1103; Bexar County; Texas  110300        37.6
2  Census Tract 1105; Bexar County; Texas  110500        25.4
3  Census Tract 1106; Bexar County; Texas  110600        37.2
4  Census Tract 1107; Bexar County; Texas  110700        47.3


In [48]:
# reecoding missing values 
final_df['median_age'] = final_df['median_age'].replace({-666666666.0: np.nan})

In [49]:
# converting 'tract' to int 
final_df['tract'] = final_df['tract'].astype('Int64')
acs_age = final_df.copy() 

In [52]:
# splitting to facilitate final merge 
bexar_age = separate_df(acs_age, 'county', '029')
travis_age = separate_df(acs_age, 'county', '453')
harris_age = separate_df(acs_age, 'county', '201')
dallas_age = separate_df(acs_age, 'county', '113')
tarrant_age = separate_df(acs_age, 'county', '439')

In [16]:
# reading in SNAP data
bexar_snap = pd.read_csv('Bexar_Data/cleaned_bexar_snap_data.csv')
travis_snap = pd.read_csv('Travis_Data/cleaned_travis_snap_data.csv')
harris_snap = pd.read_csv('Harris_Data/cleaned_harris_snap_data.csv')
dallas_snap = pd.read_csv('Dallas_Data/cleaned_dallas_snap_data.csv')
tarrant_snap = pd.read_csv('Tarrant_Data/cleaned_tarrant_snap_data.csv')

In [17]:
# separating farmers markets, converting to GDF, and projecting to appropriate CRS for distance calculations 
def convert_to_gdf(df, lon_col="Longitude", lat_col="Latitude", epsg="EPSG:4326"):
    df_clean = df.dropna(subset=[lon_col, lat_col]).copy() 
    print(f"{len(df) - len(df_clean)} missing observations dropped")
    gdf = gpd.GeoDataFrame(df_clean, geometry=gpd.points_from_xy(df_clean[lon_col], df_clean[lat_col]), crs=epsg)
    return gdf

bexar_gdf = convert_to_gdf(bexar_snap)
travis_gdf = convert_to_gdf(travis_snap)
harris_gdf = convert_to_gdf(harris_snap)
dallas_gdf = convert_to_gdf(dallas_snap)
tarrant_gdf = convert_to_gdf(tarrant_snap)

bexar_grocery_gdf = separate_df(bexar_gdf, "Store_Type", "Farmers and Markets", exclude=True)
bexar_fm_gdf = separate_df(bexar_gdf, "Store_Type", "Farmers and Markets", exclude=False)
travis_grocery_gdf = separate_df(travis_gdf, "Store_Type", "Farmers and Markets", exclude=True)
travis_fm_gdf = separate_df(travis_gdf, "Store_Type", "Farmers and Markets", exclude=False)
harris_grocery_gdf = separate_df(harris_gdf, "Store_Type", "Farmers and Markets", exclude=True)
harris_fm_gdf = separate_df(harris_gdf, "Store_Type", "Farmers and Markets", exclude=False)
dallas_grocery_gdf = separate_df(dallas_gdf, "Store_Type", "Farmers and Markets", exclude=True)
dallas_fm_gdf = separate_df(dallas_gdf, "Store_Type", "Farmers and Markets", exclude=False)
tarrant_grocery_gdf = separate_df(tarrant_gdf, "Store_Type", "Farmers and Markets", exclude=True)
tarrant_fm_gdf = separate_df(tarrant_gdf, "Store_Type", "Farmers and Markets", exclude=False)

0 missing observations dropped
0 missing observations dropped
0 missing observations dropped
0 missing observations dropped
0 missing observations dropped


In [18]:
tarrant_grocery_gdf['Store_Type'].value_counts()

Store_Type
Super Store      130
Grocery Store     97
Supermarket       96
Other             43
Name: count, dtype: int64

In [22]:
# Reading in tract shapefile
tracts_shp = gpd.read_file("food_justice/bexar_county/tl_2024_48_tract.shp")

# separating by county, projecting to appropriate CRS for distance calculations, calculating centroid, creating GDF with centroid as main geometry
bexar_tracts = separate_df(tracts_shp, "COUNTYFP", "029")
travis_tracts = separate_df(tracts_shp, "COUNTYFP", "453")
harris_tracts = separate_df(tracts_shp, "COUNTYFP", "201")
dallas_tracts = separate_df(tracts_shp, "COUNTYFP", "113")
tarrant_tracts = separate_df(tracts_shp, "COUNTYFP", "439")

# projecting to appropriate CRS for distance calculations
bexar_projected = bexar_tracts.to_crs("EPSG:2278")
travis_projected = travis_tracts.to_crs("EPSG:2277")
harris_projected = harris_tracts.to_crs("EPSG:2277")
dallas_projected = dallas_tracts.to_crs("EPSG:2276")
tarrant_projected = tarrant_tracts.to_crs("EPSG:2276")

# calculating centroids and creating gdf with centroids as main geometry
def project_centroid(gdf):
    centroids = gdf.geometry.centroid
    gdf["centroid"] = centroids
    gdf_centroids = gdf.set_geometry("centroid")
    return gdf_centroids
   
bexar_centroids_projected = project_centroid(bexar_projected)
travis_centroids_projected = project_centroid(travis_projected)
harris_centroids_projected = project_centroid(harris_projected)
dallas_centroids_projected = project_centroid(dallas_projected)
tarrant_centroids_projected = project_centroid(tarrant_projected)

# creating one gdf for distance to grocery store and one for distance to farmer's market for each county
def find_nearest(centroids_gdf, points_gdf, epsg_code, distance_colname):
    points_gdf_proj = points_gdf.to_crs(f"EPSG:{epsg_code}")
    if centroids_gdf.crs != points_gdf_proj.crs:
        centroids_proj = centroids_gdf.to_crs(f"EPSG:{epsg_code}")
    else: 
        centroids_proj = centroids_gdf.copy()
    nearest_points = gpd.sjoin_nearest(
        centroids_proj,
        points_gdf_proj,
        how="left",
        distance_col=distance_colname
    )
    return nearest_points

configs = [
    {'centroids_gdf': bexar_centroids_projected, 'points_gdf':bexar_grocery_gdf, 'epsg_code': "2278", 'distance_colname':"distance_to_nearest_grocery"},
    {'centroids_gdf': bexar_centroids_projected, 'points_gdf':bexar_fm_gdf, 'epsg_code': "2278", 'distance_colname': "distance_to_nearest_fm"},
    {'centroids_gdf': travis_centroids_projected, 'points_gdf':travis_grocery_gdf, 'epsg_code': "2277", 'distance_colname': "distance_to_nearest_grocery"},
    {'centroids_gdf': travis_centroids_projected, 'points_gdf':travis_fm_gdf, 'epsg_code': "2277", 'distance_colname': "distance_to_nearest_fm"},
    {'centroids_gdf': harris_centroids_projected, 'points_gdf':harris_grocery_gdf, 'epsg_code': "2277", 'distance_colname': "distance_to_nearest_grocery"},
    {'centroids_gdf': harris_centroids_projected, 'points_gdf':harris_fm_gdf, 'epsg_code': "2277", 'distance_colname': "distance_to_nearest_fm"},
    {'centroids_gdf': dallas_centroids_projected, 'points_gdf':dallas_grocery_gdf, 'epsg_code': "2276", 'distance_colname': "distance_to_nearest_grocery"},
    {'centroids_gdf': dallas_centroids_projected, 'points_gdf':dallas_fm_gdf, 'epsg_code': "2276", 'distance_colname': "distance_to_nearest_fm"},
    {'centroids_gdf': tarrant_centroids_projected, 'points_gdf':tarrant_grocery_gdf, 'epsg_code': "2276", 'distance_colname': "distance_to_nearest_grocery"},
    {'centroids_gdf': tarrant_centroids_projected, 'points_gdf':tarrant_fm_gdf, 'epsg_code': "2276", 'distance_colname': "distance_to_nearest_fm"},
]
distance_gdfs = [find_nearest(**config) for config in configs]
(bexar_nearest_grocery, bexar_nearest_fm, travis_nearest_grocery, travis_nearest_fm, harris_nearest_grocery, harris_nearest_fm, dallas_nearest_grocery, 
dallas_nearest_fm, tarrant_nearest_grocery, tarrant_nearest_fm) = distance_gdfs

In [23]:
# filtering to relevant columns 
columns_to_keep=['TRACTCE', 'COUNTYFP', 'County', 'GEOID', 'ALAND', 'AWATER']

grocery_gdfs = [bexar_nearest_grocery, travis_nearest_grocery, harris_nearest_grocery, dallas_nearest_grocery, tarrant_nearest_grocery]
grocery_distances = [gdf[columns_to_keep + ['distance_to_nearest_grocery']].copy() for gdf in grocery_gdfs]

fm_gdfs = [bexar_nearest_fm, travis_nearest_fm, harris_nearest_fm, dallas_nearest_fm, tarrant_nearest_fm]
fm_distances = [gdf[columns_to_keep + ['distance_to_nearest_fm']].copy() for gdf in fm_gdfs]

In [18]:
# # merging farmers market and grocery stores 
# bexar_distances = bexar_nearest_grocery.merge(bexar_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
# bexar_distances = bexar_distances.drop(columns=[col for col in bexar_distances.columns if col.endswith("_drop")])

# travis_distances = travis_nearest_grocery.merge(travis_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
# travis_distances = travis_distances.drop(columns=[col for col in travis_distances.columns if col.endswith("_drop")])

# harris_distances = harris_nearest_grocery.merge(harris_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
# harris_distances = harris_distances.drop(columns=[col for col in harris_distances.columns if col.endswith("_drop")])

# dallas_distances = dallas_nearest_grocery.merge(dallas_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
# dallas_distances = dallas_distances.drop(columns=[col for col in dallas_distances.columns if col.endswith("_drop")])

# tarrant_distances = tarrant_nearest_grocery.merge(tarrant_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
# tarrant_distances = tarrant_distances.drop(columns=[col for col in tarrant_distances.columns if col.endswith("_drop")])

In [ ]:
# Filtering Travis County Farms 
farms_to_keep = [
    "Urban Roots East Austin Farm", 
    "Urban Roots South Austin Farm",
    "HausBar Urban Farm",
    "Historic Boggy Creek Farm",
    "Green Gate Farms",
    "Gray Fox Market Garden",
    "Agua Dulce Austin",
    "Aquaflora on Evelyn",
    "Farmshare Austin",
    "New Leaf Agriculture",
    "Patchwork Farm"
]

# loading old, mis-saved file
df = pd.read_csv('final_urbanfarm_data/final_travis_farms.csv')

# filtering by value of 'name" column
travis_final = df[df['name'].isin(farms_to_keep)]

# re-saving
travis_final.to_csv('final_urbanfarm_data/final_travis_farms.csv', index=False)

In [69]:
# loading in urban farms data 
bexar_uf = pd.read_csv('final_urbanfarm_data/final_urban_farms_bexar.csv')
travis_uf = pd.read_csv('final_urbanfarm_data/final_urban_farms_travis.csv')
harris_uf = pd.read_excel('final_urbanfarm_data/harris_farms_final.xlsx')
dallas_uf = pd.read_csv('final_urbanfarm_data/final_urban_farms_dallas.csv')
tarrant_uf = pd.read_csv('final_urbanfarm_data/final_urban_farms_tarrant.csv')

# converting to gdfs
bexar_uf_gdf = convert_to_gdf(bexar_uf,'lon','lat')
travis_uf_gdf = convert_to_gdf(travis_uf, 'lon', 'lat')
harris_uf_gdf = convert_to_gdf(harris_uf, 'lon', 'lat')
dallas_uf_gdf = convert_to_gdf(dallas_uf, 'lon', 'lat')
tarrant_uf_gdf = convert_to_gdf(tarrant_uf, 'lon', 'lat') 

0 missing observations dropped
0 missing observations dropped
0 missing observations dropped
0 missing observations dropped
0 missing observations dropped


In [26]:
# dropping unnecssary columns
columns_to_keep = ['name', 'address', 'lat', 'lon', 'Website', 'profile_url', 'geometry']
bexar_uf_gdf = bexar_uf_gdf[columns_to_keep]
travis_uf_gdf = travis_uf_gdf[columns_to_keep]
harris_uf_gdf = harris_uf_gdf[columns_to_keep]
dallas_uf_gdf = dallas_uf_gdf[columns_to_keep]
tarrant_uf_gdf = tarrant_uf_gdf[columns_to_keep]

In [28]:
# calculating distances to nearest urban farm 

# projecting farms to appropriate CRS 
bexar_uf_gdf = bexar_uf_gdf.to_crs('EPSG:2278')
travis_uf_gdf = travis_uf_gdf.to_crs('EPSG:2277')
harris_uf_gdf = harris_uf_gdf.to_crs('EPSG:2277')
dallas_uf_gdf = dallas_uf_gdf.to_crs('EPSG:2276')
tarrant_uf_gdf = tarrant_uf_gdf.to_crs('EPSG:2276')

# using previously generated centroids to generate distances to nearest urban farm 
bexar_nearest_uf = find_nearest(bexar_centroids_projected, bexar_uf_gdf, "2278", "distance_to_nearest_uf")
travis_nearest_uf = find_nearest(travis_centroids_projected, travis_uf_gdf, "2277", "distance_to_nearest_uf")
harris_nearest_uf = find_nearest(harris_centroids_projected, harris_uf_gdf, "2277", "distance_to_nearest_uf")
dallas_nearest_uf = find_nearest(dallas_centroids_projected, dallas_uf_gdf, "2276", "distance_to_nearest_uf")
tarrant_nearest_uf = find_nearest(tarrant_centroids_projected, tarrant_uf_gdf, "2276", "distance_to_nearest_uf")

In [66]:
# renaming and converting 'tract' to int for easy merging 
def convert_int_rename(df, old_name, new_name):
    if old_name not in df.columns:
        print(f"'{old_name}' not in DF")
        return df.copy()
        
    df_clean = df.copy()
    df_clean = df.rename(columns={old_name:new_name}).copy()
    df_clean[new_name] = pd.to_numeric(df_clean[new_name], errors='coerce').astype('Int64')
    return df_clean

# applying function
dfs = [bexar_nearest_grocery, bexar_nearest_fm, bexar_nearest_uf, travis_nearest_grocery, travis_nearest_fm, travis_nearest_uf,
       harris_nearest_grocery, harris_nearest_fm, harris_nearest_uf, dallas_nearest_grocery, dallas_nearest_fm, dallas_nearest_uf, 
       tarrant_nearest_grocery, tarrant_nearest_fm, tarrant_nearest_uf]
(bexar_nearest_grocery, bexar_nearest_fm, bexar_nearest_uf, travis_nearest_grocery, travis_nearest_fm, travis_nearest_uf,
harris_nearest_grocery, harris_nearest_fm, harris_nearest_uf, dallas_nearest_grocery, dallas_nearest_fm, dallas_nearest_uf, 
tarrant_nearest_grocery, tarrant_nearest_fm, tarrant_nearest_uf) = [convert_int_rename(df, "TRACTCE", "tract") for df in dfs]

'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF
'TRACTCE' not in DF


In [70]:
# RETRIEVING ACS MEDIAN HOUSEHOLD INCOME AND VEHICLE ACCESS DATA FROM API

# hardcoding parameters (to change later) 
API_KEY = "3260419d0dd1c45a470edaf688621f89b71fa441"
STATE_FIPS = "48"    # Texas
COUNTY_LIST = ["029", "453", "201", "113", "439"] 

# setting API endpoint 
URL_BASE = "https://api.census.gov/data/2024/acs/acs5"
VARIABLES = "NAME,B19013_001E,B08201_002E,B08201_001E" # household income, households with no vehicle access, total households 

all_tracts_data = []

for county_fips in COUNTY_LIST:
    # building url 
    url = f"{URL_BASE}?get={VARIABLES}&for=tract:*&in=state:{STATE_FIPS}+county:{county_fips}&key={API_KEY}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        # verifying the API sent json
        content_type = response.headers.get('Content-Type', '')
        if 'application/json' in content_type:
            json_data = response.json()
            
            headers = json_data[0]
            rows = json_data[1:]
            county_df = pd.DataFrame(rows, columns=headers)
            all_tracts_data.append(county_df)
        else:
            # print out text error if necessary
            print(f"\n[Census API Error] Server returned text instead of data for county {county_fips}:")
            print(response.text.strip())

    except requests.exceptions.RequestException as e:
        print(f"Network or HTTP error for county {county_fips}: {e}")

# combining results
if all_tracts_data:
    acs_income_vehicle = pd.concat(all_tracts_data, ignore_index=True)
    
    # converting and cleaning results
    acs_income_vehicle["B19013_001E"] = pd.to_numeric(acs_income_vehicle["B19013_001E"], errors='coerce')
    acs_income_vehicle["B08201_002E"] = pd.to_numeric(acs_income_vehicle["B08201_002E"], errors='coerce')
    acs_income_vehicle["B08201_001E"] = pd.to_numeric(acs_income_vehicle["B08201_001E"], errors='coerce')
    acs_income_vehicle = acs_income_vehicle.rename(columns={"B19013_001E": "median_income", "B08201_002E": "no_vehicle", "B08201_001E": "total_households"})
    
    print("\n--- SUCCESS! First 5 rows: ---")
    print(acs_income_vehicle[["NAME", "tract", "median_income", "no_vehicle", "total_households"]].head())
else:
    print("\nNo data was collected. Read the API text errors above to see why.")


--- SUCCESS! First 5 rows: ---
                                     NAME   tract  median_income  no_vehicle  \
0  Census Tract 1101; Bexar County; Texas  110100          52109         661   
1  Census Tract 1103; Bexar County; Texas  110300          50189         201   
2  Census Tract 1105; Bexar County; Texas  110500          26932         456   
3  Census Tract 1106; Bexar County; Texas  110600          23241         510   
4  Census Tract 1107; Bexar County; Texas  110700          25313         151   

   total_households  
0              2659  
1              1139  
2              1115  
3              1482  
4               534  


In [82]:
# recoding missing values
acs_income_vehicle = acs_income_vehicle.replace({-666666666.0: np.nan})

# converting 'tract' number to int 
acs_income_vehicle[['tract', 'state', 'county']] = acs_income_vehicle[['tract', 'state', 'county']].astype('Int64')

In [83]:
# creating pct of households with no vehicle variable
acs_income_vehicle['pct_no_vehicle'] = acs_income_vehicle['no_vehicle'] / acs_income_vehicle['total_households'] * 100 

In [84]:
# splitting by county to facilitate final merge
bexar_income_vehicle = separate_df(acs_income_vehicle, "county", 29)
travis_income_vehicle = separate_df(acs_income_vehicle, "county", 453)
harris_income_vehicle = separate_df(acs_income_vehicle, "county", 201)
dallas_income_vehicle = separate_df(acs_income_vehicle, "county", 113)
tarrant_income_vehicle = separate_df(acs_income_vehicle, "county", 439)

In [85]:
# merging all of the dataframes into one for modeling 
merge_key = 'tract'

# defining merge function
def smart_merge(left, right):
    # identifying overlapping columns, excluding merge key 
    overlapping_cols = [col for col in right.columns if col in left.columns and col != merge_key]
    # dropping all overlapping columns from 'right' df
    right_cleaned = right.drop(columns=overlapping_cols)
    # performing the merge 
    return pd.merge(left, right_cleaned, on=merge_key, how='outer')

# merging all of the counties 
bexar_dfs = [bexar_EJI, bexar_age, bexar_nearest_grocery, bexar_nearest_fm, bexar_nearest_uf, bexar_income_vehicle]
bexar_merged = reduce(smart_merge, bexar_dfs)
travis_dfs = [travis_EJI, travis_age, travis_nearest_grocery, travis_nearest_fm,travis_nearest_uf, travis_income_vehicle]
travis_merged = reduce(smart_merge, travis_dfs)
harris_dfs = [harris_EJI, harris_age, harris_nearest_grocery, harris_nearest_fm, harris_nearest_uf, harris_income_vehicle]
harris_merged = reduce(smart_merge, harris_dfs)
dallas_dfs = [dallas_EJI, dallas_age, dallas_nearest_grocery, dallas_nearest_fm, dallas_nearest_uf, dallas_income_vehicle]
dallas_merged = reduce(smart_merge, dallas_dfs)
tarrant_dfs = [tarrant_EJI, tarrant_age, tarrant_nearest_grocery, tarrant_nearest_fm, tarrant_nearest_uf, tarrant_income_vehicle]
tarrant_merged = reduce(smart_merge, tarrant_dfs)

In [88]:
# cleaning up columns 
columns_to_keep = ['COUNTY', 'GEOID', 'tract', 'walking_ind', 'walkind_inv_perc', 'total_pop',
                             'total_pop_moe', 'below_200_fed_poverty_percentage', 'no_hs_diploma', 'uninsured', 'E_CHD', 'E_DIABETES', 
                             'E_AFAM', 'E_ASIAN', 'E_HISP', 'median_age', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm', 
                             'geometry', 'distance_to_nearest_uf', 'median_income', 'pct_no_vehicle']
bexar_merged = bexar_merged[columns_to_keep]
travis_merged = travis_merged[columns_to_keep]
harris_merged = harris_merged[columns_to_keep]
dallas_merged = dallas_merged[columns_to_keep]
tarrant_merged = tarrant_merged[columns_to_keep]

# merging and writing to files 
bexar_merged_gdf = gpd.GeoDataFrame(bexar_merged, geometry='geometry')
bexar_merged_gdf.to_file("modeling_data/bexar_modeling_final.gpkg", driver="GPKG")

travis_merged_gdf = gpd.GeoDataFrame(travis_merged, geometry='geometry')
travis_merged_gdf.to_file("modeling_data/travis_modeling_final.gpkg", driver="GPKG")

harris_merged_gdf = gpd.GeoDataFrame(harris_merged, geometry='geometry')
harris_merged_gdf.to_file("modeling_data/harris_modeling_final.gpkg", driver="GPKG")

dallas_merged_gdf = gpd.GeoDataFrame(dallas_merged, geometry='geometry')
dallas_merged_gdf.to_file("modeling_data/dallas_modeling_final.gpkg", driver="GPKG")

tarrant_merged_gdf = gpd.GeoDataFrame(tarrant_merged, geometry='geometry')
tarrant_merged_gdf.to_file("modeling_data/tarrant_modeling_final.gpkg", driver="GPKG")

In [66]:
# creating master dataset 
bexar_merged_gdf = gpd.read_file("modeling_data/bexar_modeling_final.gpkg").to_crs("EPSG:3857")
travis_merged_gdf = gpd.read_file("modeling_data/travis_modeling_final.gpkg").to_crs("EPSG:3857")
harris_merged_gdf = gpd.read_file("modeling_data/harris_modeling_final.gpkg").to_crs("EPSG:3857")
dallas_merged_gdf = gpd.read_file("modeling_data/dallas_modeling_final.gpkg").to_crs("EPSG:3857")
tarrant_merged_gdf = gpd.read_file("modeling_data/tarrant_modeling_final.gpkg").to_crs("EPSG:3857")
county_dfs = [bexar_merged_gdf, travis_merged_gdf, harris_merged_gdf, dallas_merged_gdf, tarrant_merged_gdf]
modeling_data_final = pd.concat(county_dfs, ignore_index=True)

In [67]:
modeling_data_final.to_file("modeling_data/modeling_data_final.gpkg", driver="GPKG")